<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

Parameter order follows the module tree, which is the order the Paddle exporter emitted Conv, ConvTranspose, BatchNormalization, MatMul, and the decomposed LayerNorm scale and bias. Paddle linear weights are stored as `(in, out)`. MAX linear weights are `(out, in)`.

A bias is the next Add, skipping Identity nodes, whose right-hand side is a learned vector. A scalar add is not a bias. LayerNorm scale is the same kind of vector sitting on a Mul, and the bias is the Add that follows it.

In [7]:
#| echo: false
#| output: asis
show_doc(arrays_from_graph)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/weights.py#L54){target="_blank" style="float:right; font-size:smaller"}

### arrays_from_graph

```python
def arrays_from_graph(
    graph:Graph
)->list[np.ndarray]:
```

*Learned tensors in module state-dict order.*

Conv weights stay in the checkpoint's layout. [`conv2d`](https://alix559.github.io/solid_lens/conv.html#conv2d) was built with `permute=True` so that layout is already OIHW. MatMul is the exception: the stored matrix is `(in, out)` and MAX wants `(out, in)`, so it is transposed here.

In [8]:
import numpy as np
from fastcore.test import test_eq
from solid_lens.onnx import Graph, Node

weight = np.arange(6, dtype=np.float32).reshape(2, 3)
graph = Graph()
graph.initializers["w"] = weight
graph.initializers["b"] = np.array([10, 20, 30], np.float32)
matmul = Node()
matmul.op_type = "MatMul"
matmul.name = "fc"
matmul.inputs = ["x", "w"]
add = Node()
add.op_type = "Add"
add.inputs = ["y", "b"]
graph.nodes = [matmul, add]
arrays = arrays_from_graph(graph)
test_eq(arrays[0].tolist(), weight.T.tolist())
test_eq(arrays[1].tolist(), [10.0, 20.0, 30.0])
arrays[0]

array([[0., 3.],
       [1., 4.],
       [2., 5.]], dtype=float32)

In [10]:
#| echo: false
#| output: asis
show_doc(checkpoint_arrays)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/weights.py#L97){target="_blank" style="float:right; font-size:smaller"}

### checkpoint_arrays

```python
def checkpoint_arrays(
    path:Path
)->list[np.ndarray]:
```

*Learned tensors in an ONNX checkpoint, in module state-dict order.*

MAX walks `layer_weights` and then `sublayers`. Defining a submodule later in `__init__` puts its tensors later. That is the same order as the exporter, which is why the layer notebooks keep the attribute order they do.

In [13]:
#| echo: false
#| output: asis
show_doc(load_checkpoint)

---

[source](https://github.com/alix559/solid_lens/blob/main/solid_lens/weights.py#L115){target="_blank" style="float:right; font-size:smaller"}

### load_checkpoint

```python
def load_checkpoint(
    module, path:Path
)->None:
```

A count mismatch and a shape mismatch are both errors. There is no partial load. If a new layer is inserted in the wrong place, this is the function that says so.